# CG HEWL — FRESEAN parallel benchmark

This notebook provides a brief tutorial on how to use pyFRESEAN parallel workflow settings. It uses the same CG trajectory and spectral parameters as in [02_AA_CG-hewl-solution-300K.ipynb](02_AA_CG-hewl-solution-300K.ipynb), but focuses only on the CG FRESEAN step.

### What is being parallelized?

After the trajectory is read, pyFRESEAN runs post-processing in `_conclude` in three main phases:

| Phase key | What it does |
|-----------|----------------|
| `velocity_fft` | FFT-based velocity spectra from collected frames |
| `corr_matrix` | Build the frequency-dependent correlation matrix (often the most expensive step) |
| `eigen` | Diagonalize the correlation matrix at each frequency (modes) |

The `parallel` argument on `FRESEAN(...)` controls **only these phases**, with 2 types of parallelization options implemented in pyFRESEAN:

- **`n_jobs`** — Python `ThreadPoolExecutor` workers (good for tiling work in the correlation matrix).
- **`omp_threads`** — BLAS/OpenMP threads inside NumPy/SciPy for that phase (good for large linear algebra in FFT and eigen steps).

Analysis routines in pyFRESEAN are built as derivatives of the MDAnalysis `AnalysisBase` class, so they provide `multiprocessing` based parallel reading of trajectories. However, for this notebook, we do **not** change frame-parallel trajectory reading here (`run(n_workers=1)` by default). That keeps the comparison focused on `_conclude` threading.

### How to use this notebook

1. Run from the `examples/` directory so `input_data/` resolves.
2. Run **§1–2** once (CPU count, then load CG + `Align`, create `BENCH_RESULTS`).
3. Run each **case** cell in order (or skip cases you do not need). Each cell appends one dict to `BENCH_RESULTS`.
4. Run the **Results table** cell to build final table.

For quicker tests, lower `STOP` in setup (e.g. `200` instead of `1000`). Avoid running other heavy jobs on the same machine while timing.

## 1. CPUs available to this kernel

No single Python call reports “CPUs allocated to this job” in every environment. What you see depends on how the kernel was started.  We show here below some of the common methods. The user may enter a custom number of CPUs based on their system's configuration.

| Source | Meaning |
|--------|--------|
| `os.cpu_count()` | Logical CPUs on the **machine** (e.g. 128 on a big node) — often **too large** under SLURM. |
| `len(os.sched_getaffinity(0))` (Linux) | CPUs this **process** is allowed to run on — correct when SLURM/cgroups bind the kernel. |
| `SLURM_CPUS_PER_TASK` | CPUs Slurm assigned to your job step (when set). |

The next cell picks `N_CPU` in this order: manual override → Slurm env → affinity smaller than machine → affinity. If you still see the full node (common on a **login node** or an unbound Jupyter server), set **`N_CPU_OVERRIDE`** to your allocation (e.g. `8`).

In [1]:
import os
import time
from pathlib import Path

import pandas as pd

from pyfresean import Align, CoarseGrain, FRESEAN
from pyfresean.benchmark_keys import (
    BENCH_T_CORR_MATRIX,
    BENCH_T_EIGEN,
    BENCH_T_FRESEAN_TOTAL,
    BENCH_T_READ_TRAJ,
    BENCH_T_VDOS,
    BENCH_T_VELOCITY_SPECTRA,
)

N_CPU_OVERRIDE = None  # e.g. 8 — set if auto-detect still shows the whole node

machine_cpus = os.cpu_count() or 1
affinity_cpus = (
    len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else machine_cpus
)
slurm_cpus = os.environ.get("SLURM_CPUS_PER_TASK")
slurm_cpus = int(slurm_cpus) if slurm_cpus is not None else None

if N_CPU_OVERRIDE is not None:
    N_CPU = int(N_CPU_OVERRIDE)
elif slurm_cpus is not None:
    N_CPU = slurm_cpus
elif affinity_cpus < machine_cpus:
    N_CPU = affinity_cpus
else:
    N_CPU = affinity_cpus

print(f"kernel pid={os.getpid()}")
print(f"  os.cpu_count() (machine):     {machine_cpus}")
print(f"  sched_getaffinity (process):  {affinity_cpus}")
print(f"  SLURM_CPUS_PER_TASK:          {slurm_cpus}")
print(f"  N_CPU used in benchmarks:     {N_CPU}")

kernel pid=1375609
  os.cpu_count() (machine):     128
  sched_getaffinity (process):  4
  SLURM_CPUS_PER_TASK:          4
  N_CPU used in benchmarks:     4


## 2. Trajectory, parameters, and results list

Load the coarse-grained HEWL universe, apply `Align`, and create an empty **`BENCH_RESULTS`** list. Each benchmark cell appends one dict with `case`, six parallel fields (`*_n_jobs`, `*_n_omp` per phase, default `1`), `wall_s`, and phase timings from `run(benchmark=True)`.

In [2]:
INPUT_DATA = Path("input_data") / "MD-HEWL-303K"
START, STOP, STEP = 0, 1000, 1  # reduce STOP for quicker tests
N_CORR, DT, SIGMA = 100, 0.02, 10.0

topol = INPUT_DATA / "output/topol_prot.tpr"
traj = INPUT_DATA / "output/sample-NPT_prot_pbc.trr"
if not topol.exists() or not traj.exists():
    raise FileNotFoundError("Missing HEWL files — see input_data/README.md")

cg, u_cg = CoarseGrain.cg_universe(
    (topol, traj),
    select="protein",
    start=START,
    stop=STOP,
    step=STEP,
    output_cg_topology=None,
    output_cg_trajectory=None,
    centered_mode="ref",
    in_memory=True,
)
u_cg.trajectory[0]
ref_cg = u_cg.atoms.positions.copy()
u_cg.trajectory.add_transformations(
    Align(u_cg.atoms, reference_positions=ref_cg, place_com_in_box=False),
)
print(f"CG beads={cg.mapping.n_beads}, frames={len(u_cg.trajectory)}")

BENCH_RESULTS = []

CG beads=246, frames=1000


## 3. Baseline — serial `_conclude` (no `parallel`)

If you omit `parallel`, every phase uses the defaults: `n_jobs=1`, `omp_threads=1`. This is our reference timing we will compare every other case against.

`wall_s` is end-to-end time for constructing `FRESEAN` and calling `run(...)`. The `timings` dict breaks that into trajectory read vs each `_conclude` phase and sub-phase. Detailed list of phase/sub-phase keys is in `pyfresean.benchmark_keys`.

In [3]:
t0 = time.perf_counter()
analysis = FRESEAN(
    u_cg,
    select="all",
    n_constraints=int(cg.mapping.n_constraints),
    n_corr=N_CORR,
    dt=DT,
    sigma=SIGMA,
)
timings = analysis.run(start=START, stop=STOP, step=STEP, benchmark=True)
wall_s = time.perf_counter() - t0

row = {
    "case": "baseline_serial",
    "velocity_fft_n_jobs": 1,
    "velocity_fft_n_omp": 1,
    "corr_matrix_n_jobs": 1,
    "corr_matrix_n_omp": 1,
    "eigen_n_jobs": 1,
    "eigen_n_omp": 1,
    "wall_s": wall_s,
    **timings,
}
BENCH_RESULTS.append(row)
print(row)

{'case': 'baseline_serial', 'velocity_fft_n_jobs': 1, 'velocity_fft_n_omp': 1, 'corr_matrix_n_jobs': 1, 'corr_matrix_n_omp': 1, 'eigen_n_jobs': 1, 'eigen_n_omp': 1, 'wall_s': 17.537976627005264, 'bench_t_read_traj': 0.03520182194188237, 'bench_t_velocity_spectra': 0.028272800846025348, 'bench_t_corr_matrix': 9.115870722103864, 'bench_t_vdos': 0.05794452107511461, 'bench_t_eigen': 8.299664946040139, 'bench_t_fresean_total': 17.536954812007025}


## 4. `velocity_fft` only — BLAS threading (`omp_threads`)

Here we only speed up the **velocity FFT** phase. We set `omp_threads=N_CPU` and keep `n_jobs=1` (no extra Python thread pool).

Correlation matrix and eigen phases stay at defaults. Use this cell to see whether velocity FFT dominates your workload and can help speedup things. Depending on the system size and the size of the timeseries, this may or may not help speed up the calculation.

In [4]:
t0 = time.perf_counter()
analysis = FRESEAN(
    u_cg,
    select="all",
    n_constraints=int(cg.mapping.n_constraints),
    n_corr=N_CORR,
    dt=DT,
    sigma=SIGMA,
    parallel={
        "velocity_fft": {"n_jobs": 1, "omp_threads": N_CPU},
    },
)
timings = analysis.run(start=START, stop=STOP, step=STEP, benchmark=True)
wall_s = time.perf_counter() - t0

row = {
    "case": "velocity_fft_only_omp",
    "velocity_fft_n_jobs": 1,
    "velocity_fft_n_omp": N_CPU,
    "corr_matrix_n_jobs": 1,
    "corr_matrix_n_omp": 1,
    "eigen_n_jobs": 1,
    "eigen_n_omp": 1,
    "wall_s": wall_s,
    **timings,
}
BENCH_RESULTS.append(row)
print(row)

{'case': 'velocity_fft_only_omp', 'velocity_fft_n_jobs': 1, 'velocity_fft_n_omp': 4, 'corr_matrix_n_jobs': 1, 'corr_matrix_n_omp': 1, 'eigen_n_jobs': 1, 'eigen_n_omp': 1, 'wall_s': 17.302595915971324, 'bench_t_read_traj': 0.01846876204945147, 'bench_t_velocity_spectra': 0.005881186109036207, 'bench_t_corr_matrix': 8.896180094219744, 'bench_t_vdos': 0.057428197003901005, 'bench_t_eigen': 8.200321533018723, 'bench_t_fresean_total': 17.178279772400856}


## 5. `corr_matrix` only — thread pool (`n_jobs`)

The correlation matrix build is often parallelized with **`n_jobs`** (several tiles at once). Set **`omp_threads=1`** so BLAS does not spawn many threads *inside* each worker (oversubscription slows things down).

Compare `bench_t_corr_matrix` in the printed row to the baseline — this is usually where hybrid setups win most.

In [5]:
t0 = time.perf_counter()
analysis = FRESEAN(
    u_cg,
    select="all",
    n_constraints=int(cg.mapping.n_constraints),
    n_corr=N_CORR,
    dt=DT,
    sigma=SIGMA,
    parallel={
        "corr_matrix": {"n_jobs": N_CPU, "omp_threads": 1},
    },
)
timings = analysis.run(start=START, stop=STOP, step=STEP, benchmark=True)
wall_s = time.perf_counter() - t0

row = {
    "case": "corr_matrix_only_threads",
    "velocity_fft_n_jobs": 1,
    "velocity_fft_n_omp": 1,
    "corr_matrix_n_jobs": N_CPU,
    "corr_matrix_n_omp": 1,
    "eigen_n_jobs": 1,
    "eigen_n_omp": 1,
    "wall_s": wall_s,
    **timings,
}
BENCH_RESULTS.append(row)
print(row)

{'case': 'corr_matrix_only_threads', 'velocity_fft_n_jobs': 1, 'velocity_fft_n_omp': 1, 'corr_matrix_n_jobs': 4, 'corr_matrix_n_omp': 1, 'eigen_n_jobs': 1, 'eigen_n_omp': 1, 'wall_s': 13.774365493096411, 'bench_t_read_traj': 0.021280772052705288, 'bench_t_velocity_spectra': 0.012050024000927806, 'bench_t_corr_matrix': 5.21068927901797, 'bench_t_vdos': 0.0625447710044682, 'bench_t_eigen': 8.337488449877128, 'bench_t_fresean_total': 13.6440532959532}


## 6. `eigen` only — BLAS threading (`omp_threads`)

Mode diagonalization at each frequency bin can use multi-threaded linear algebra. Same pattern as the FFT case: `omp_threads=N_CPU`, `n_jobs=1`.

For small CG systems the eigen step may be a modest fraction of total time; for larger systems or many bins it can matter more.

In [6]:
t0 = time.perf_counter()
analysis = FRESEAN(
    u_cg,
    select="all",
    n_constraints=int(cg.mapping.n_constraints),
    n_corr=N_CORR,
    dt=DT,
    sigma=SIGMA,
    parallel={
        "eigen": {"n_jobs": 1, "omp_threads": N_CPU},
    },
)
timings = analysis.run(start=START, stop=STOP, step=STEP, benchmark=True)
wall_s = time.perf_counter() - t0

row = {
    "case": "eigen_only_omp",
    "velocity_fft_n_jobs": 1,
    "velocity_fft_n_omp": 1,
    "corr_matrix_n_jobs": 1,
    "corr_matrix_n_omp": 1,
    "eigen_n_jobs": 1,
    "eigen_n_omp": N_CPU,
    "wall_s": wall_s,
    **timings,
}
BENCH_RESULTS.append(row)
print(row)

{'case': 'eigen_only_omp', 'velocity_fft_n_jobs': 1, 'velocity_fft_n_omp': 1, 'corr_matrix_n_jobs': 1, 'corr_matrix_n_omp': 1, 'eigen_n_jobs': 1, 'eigen_n_omp': 4, 'wall_s': 17.02024207287468, 'bench_t_read_traj': 0.016651486046612263, 'bench_t_velocity_spectra': 0.00570943090133369, 'bench_t_corr_matrix': 9.012089323019609, 'bench_t_vdos': 0.062341507989913225, 'bench_t_eigen': 7.804420472821221, 'bench_t_fresean_total': 16.90121222077869}


## 7. Hybrid (recommended) — combine the strategies

- **Thread pool** for `corr_matrix` (`n_jobs=N_CPU`, `omp_threads=1`).
- **BLAS threads** for `velocity_fft` and `eigen` (`n_jobs=1`, `omp_threads=N_CPU`).

Phases run one after another, so each phase can use a different parallelism model.

In [7]:
t0 = time.perf_counter()
analysis = FRESEAN(
    u_cg,
    select="all",
    n_constraints=int(cg.mapping.n_constraints),
    n_corr=N_CORR,
    dt=DT,
    sigma=SIGMA,
    parallel={
        "velocity_fft": {"n_jobs": 1, "omp_threads": N_CPU},
        "corr_matrix": {"n_jobs": N_CPU, "omp_threads": 1},
        "eigen": {"n_jobs": 1, "omp_threads": N_CPU},
    },
)
timings = analysis.run(start=START, stop=STOP, step=STEP, benchmark=True)
wall_s = time.perf_counter() - t0

row = {
    "case": "hybrid_recommended",
    "velocity_fft_n_jobs": 1,
    "velocity_fft_n_omp": N_CPU,
    "corr_matrix_n_jobs": N_CPU,
    "corr_matrix_n_omp": 1,
    "eigen_n_jobs": 1,
    "eigen_n_omp": N_CPU,
    "wall_s": wall_s,
    **timings,
}
BENCH_RESULTS.append(row)
print(row)

{'case': 'hybrid_recommended', 'velocity_fft_n_jobs': 1, 'velocity_fft_n_omp': 4, 'corr_matrix_n_jobs': 4, 'corr_matrix_n_omp': 1, 'eigen_n_jobs': 1, 'eigen_n_omp': 4, 'wall_s': 12.728927474003285, 'bench_t_read_traj': 0.020823490107432008, 'bench_t_velocity_spectra': 0.012047308031469584, 'bench_t_corr_matrix': 5.17829286493361, 'bench_t_vdos': 0.1325623181182891, 'bench_t_eigen': 7.257625604048371, 'bench_t_fresean_total': 12.601351585239172}


## 8. Results table

One row per benchmark case: **`case`**, parallel settings (`n_jobs` / `n_omp` under each phase header), **`wall_s`**, then phase timings from `run(benchmark=True)` (seconds).

You may notice that parallelization is not always beneficial for certian phases of the analysis. This however changes with larger system sizes and the size of the timeseries, and thus settings must be chosen by the user in accordance with that.

Any variation observed across similar runs or phases of runs is due to the small use-case demonstarted here. Users are advised to test this out on larger systems and timeseries (by tuning `START`, `STOP`, and `STEP`) to get a better understanding of the parallelization. Fruther, quantitative timing benchmarks must done in repeats to get a reliable estimate of parallelized performance.

In [8]:
bench_df = pd.DataFrame(BENCH_RESULTS)

parallel = bench_df[
    [
        "velocity_fft_n_jobs",
        "velocity_fft_n_omp",
        "corr_matrix_n_jobs",
        "corr_matrix_n_omp",
        "eigen_n_jobs",
        "eigen_n_omp",
    ]
].copy()
parallel.columns = pd.MultiIndex.from_tuples(
    [
        ("velocity_fft", "n_jobs"),
        ("velocity_fft", "n_omp"),
        ("corr_matrix", "n_jobs"),
        ("corr_matrix", "n_omp"),
        ("eigen", "n_jobs"),
        ("eigen", "n_omp"),
    ]
)

time_keys = [
    BENCH_T_VELOCITY_SPECTRA,
    BENCH_T_CORR_MATRIX,
    BENCH_T_EIGEN,
    BENCH_T_FRESEAN_TOTAL,
]
times = bench_df[[k for k in time_keys if k in bench_df.columns]].round(3)

bench_table = pd.concat(
    [
        bench_df[["case"]],
        parallel,
        bench_df[["wall_s"]].round(3),
        times,
    ],
    axis=1,
)
bench_table

,case,"(velocity_fft, n_jobs)","(velocity_fft, n_omp)","(corr_matrix, n_jobs)","(corr_matrix, n_omp)","(eigen, n_jobs)","(eigen, n_omp)",wall_s,bench_t_velocity_spectra,bench_t_corr_matrix,bench_t_eigen,bench_t_fresean_total
0,baseline_serial,1,1,1,1,1,1,17.538,0.028,9.116,8.300,17.537
1,velocity_fft_only_omp,1,4,1,1,1,1,17.303,0.006,8.896,8.200,17.178
2,corr_matrix_only_threads,1,1,4,1,1,1,13.774,0.012,5.211,8.337,13.644
3,eigen_only_omp,1,1,1,1,1,4,17.020,0.006,9.012,7.804,16.901
4,hybrid_recommended,1,4,4,1,1,4,12.729,0.012,5.178,7.258,12.601
